In [32]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

import swifter

import nltk
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

import string
import warnings

# Download NLTK data
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
# nltk.download('punkt_tab')
# nltk.download('averaged_perceptron_tagger_eng')

nltk.download('stopwords')
nltk.download('wordnet')

warnings.filterwarnings('ignore')



[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Bangkit\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Bangkit\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Bangkit\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Bangkit\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Bangkit\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Bangkit\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-d

In [3]:
train = pd.read_csv('./Data/train_labelled.csv')
val = pd.read_csv('./Data/val_labelled.csv')
test = pd.read_csv('./Data/test_labelled.csv')

In [19]:
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN
    
# Preprocessing function
def preprocess_text(text):
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    
    # Tokenize text
    tokens = word_tokenize(text.lower(), language='english')
    
    # Get POS tags
    pos_tags = pos_tag(tokens)

    # Remove punctuation and stopwords, and lemmatize
    tokens = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags 
              if word not in stop_words and word not in string.punctuation]
    return ' '.join(tokens)


In [29]:
X_train = train['title'].swifter.apply(preprocess_text).to_list()
y_train = np.array(train['sentiment_label'])

X_test = test['title'].swifter.apply(preprocess_text).to_list()
y_test = np.array(test['sentiment_label'])


Pandas Apply: 100%|██████████| 23239/23239 [00:30<00:00, 751.51it/s]


In [31]:

# Hyperparameters
vocab_size = 10000
embedding_dim = 16
max_length = 20
trunc_type = 'post'
padding_type = 'post'
oov_token = "<OOV>"

# Tokenization and padding
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(X_train)
word_index = tokenizer.word_index
sequences = tokenizer.texts_to_sequences(X_train)
X_train_ = pad_sequences(sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

In [34]:

# Build the RNN model
model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_length),
    SimpleRNN(128, activation='tanh', return_sequences=False),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
model.fit(X_train_, y_train, epochs=50, batch_size=500, verbose=2)


Epoch 1/50
375/375 - 13s - loss: 0.6924 - accuracy: 0.5145 - 13s/epoch - 34ms/step
Epoch 2/50
375/375 - 9s - loss: 0.6870 - accuracy: 0.5392 - 9s/epoch - 24ms/step
Epoch 3/50
375/375 - 9s - loss: 0.6743 - accuracy: 0.5695 - 9s/epoch - 24ms/step
Epoch 4/50
375/375 - 9s - loss: 0.6513 - accuracy: 0.6007 - 9s/epoch - 24ms/step
Epoch 5/50
375/375 - 9s - loss: 0.6238 - accuracy: 0.6249 - 9s/epoch - 23ms/step
Epoch 6/50
375/375 - 9s - loss: 0.5965 - accuracy: 0.6439 - 9s/epoch - 24ms/step
Epoch 7/50
375/375 - 9s - loss: 0.5740 - accuracy: 0.6563 - 9s/epoch - 24ms/step
Epoch 8/50
375/375 - 9s - loss: 0.5550 - accuracy: 0.6646 - 9s/epoch - 25ms/step
Epoch 9/50
375/375 - 10s - loss: 0.5380 - accuracy: 0.6734 - 10s/epoch - 26ms/step
Epoch 10/50
375/375 - 10s - loss: 0.5250 - accuracy: 0.6782 - 10s/epoch - 26ms/step
Epoch 11/50
375/375 - 10s - loss: 0.5150 - accuracy: 0.6817 - 10s/epoch - 26ms/step
Epoch 12/50
375/375 - 10s - loss: 0.5051 - accuracy: 0.6875 - 10s/epoch - 27ms/step
Epoch 13/50
375

In [50]:

# Example inference
new_sequences = tokenizer.texts_to_sequences(X_test)
X_test_ = pad_sequences(new_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

y_pred_proba = model.predict(X_test_)
y_pred = (y_pred_proba > 0.5).astype(int)


727/727 [==============================] - 5s 6ms/step


In [51]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.51      0.43      0.47     11976
           1       0.48      0.56      0.52     11263

    accuracy                           0.50     23239
   macro avg       0.50      0.50      0.49     23239
weighted avg       0.50      0.50      0.49     23239

